# PDDL Pipeline Diagnostic

This notebook runs **each step of the PDDL pipeline** separately to diagnose
where the process is producing garbage text (e.g., OCR noise like "Aaun").

## Analyzed flow:
1. Toolbox: dataset discovery → sample image → artifact retrieval
2. Toolbox: structure extraction via Docling
3. Builder: `ProcessingManifest` construction
4. Manifest inspection (elements, types, texts)
5. Mostly-visual manifest detection
6. Image enrichment (VisionAgent)
7. Final text rendering
8. Local cache inspection

> The image is obtained dynamically from the `acessilia-dataset` via the Toolbox dataset API — no hardcoded paths needed.

In [18]:
# ═══════════════════════════════════════════════════════════════
# Cell 1 — Setup: imports, project root, image from dataset
# ═══════════════════════════════════════════════════════════════
import sys
import json
import os
from pathlib import Path
from pprint import pprint

# ── Detect project root ──
_PROJECT_CANDIDATES = [
    Path.cwd().resolve(),
]
PROJECT_ROOT = None
for cand in filter(None, _PROJECT_CANDIDATES):
    if (cand / "backend").is_dir():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "backend").is_dir():
            PROJECT_ROOT = p
            break
        p = p.parent

if PROJECT_ROOT and str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Dataset configuration ──
# The image is fetched from the acessilia-dataset via the Toolbox dataset API.
# No local file path is hardcoded.
DATASET_ID = "acessilia-dataset"
DATASET_SPLIT = "input"
DATASET_REVISION = "main"
DATASET_ITEM_ID = "005"          # item ID in the dataset (005.jpeg)
DATASET_ARTIFACT_PATH = "005.jpeg"

print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📦 Dataset:      {DATASET_ID}")
print(f"   Split:        {DATASET_SPLIT}")
print(f"   Revision:     {DATASET_REVISION}")
print(f"   Item:         {DATASET_ITEM_ID}")
print(f"   Artifact:     {DATASET_ARTIFACT_PATH}")

📁 Project root: /Users/akira/dados/sync/dev/acessilia
📦 Dataset:      acessilia-dataset
   Split:        input
   Revision:     main
   Item:         005
   Artifact:     005.jpeg


In [19]:
# ═══════════════════════════════════════════════════════════════
# Cell 2 — Environment settings
# ═══════════════════════════════════════════════════════════════
from backend.config.settings import settings

print("🔧 Relevant settings:")
print(f"   TOOLBOX_BASE_URL:       {settings.toolbox_base_url}")
print(f"   TOOLBOX_PROVIDER:       {settings.toolbox_provider}")
print(f"   TOOLBOX_USE_ARTIFACT_STORE: {settings.toolbox_use_artifact_store}")
print(f"   TOOLBOX_USE_REMOTE_CACHE:   {settings.toolbox_use_remote_cache}")
print(f"   AI_CLIENT:              {settings.ai_client}")
print(f"   OPENROUTER_MODEL:       {settings.openrouter_model}")
print(f"   PIPELINE_ENGINE:        {settings.pipeline_engine}")
print(f"   STRUCTURER:             {settings.structurer}")
print(f"   PDDL_PLANNER_BACKEND:   {settings.pddl_planner_backend}")

🔧 Relevant settings:
   TOOLBOX_BASE_URL:       http://localhost:8002
   TOOLBOX_PROVIDER:       docling
   TOOLBOX_USE_ARTIFACT_STORE: True
   TOOLBOX_USE_REMOTE_CACHE:   True
   AI_CLIENT:              openrouter
   OPENROUTER_MODEL:       qwen/qwen3-vl-8b-instruct
   PIPELINE_ENGINE:        pddl
   STRUCTURER:             toolbox
   PDDL_PLANNER_BACKEND:   internal


In [20]:
# ═══════════════════════════════════════════════════════════════
# Cell 3 — Explore datasets and retrieve the sample image
# ═══════════════════════════════════════════════════════════════
from backend.tools.toolbox_client import ToolboxClient

client = ToolboxClient()
print(f"🔗 Toolbox base URL: {client.base_url}")
print(f"🔗 Toolbox provider: {client.provider}")

# List available datasets
datasets = await client.list_datasets()
print(f"\n📚 Available datasets: {len(datasets)}")
for ds in datasets:
    print(f"   - {ds['id']}: {ds.get('name', ds['id'])}")

# Describe the acessilia-dataset
info = await client.describe_dataset(DATASET_ID, revision=DATASET_REVISION)
print(f"\n📖 Dataset info:")
print(f"   Name:        {info.get('name')}")
print(f"   Description: {info.get('description', '')[:120]}...")
print(f"   Splits:      {info.get('splits', [])}")
print(f"   Items:       {info.get('item_count', '?')}")

# List splits
splits = await client.list_splits(DATASET_ID, revision=DATASET_REVISION)
print(f"\n📂 Splits:")
for sp in splits:
    print(f"   - {sp['id']}: {sp.get('item_count', '?')} items")

# List items in the input split
items = await client.list_items(
    DATASET_ID, DATASET_SPLIT,
    revision=DATASET_REVISION,
    limit=10,
)
print(f"\n📄 Items in '{DATASET_SPLIT}' split (first {len(items)}):")
for item in items:
    print(f"   - {item['id']}: {item.get('artifact_count', '?')} artifacts")

# Inspect item 005 to find actual artifact paths
print(f"\n🔍 Inspecting item '{DATASET_ITEM_ID}' to find artifact paths...")
item_detail = await client.get_item(
    DATASET_ID, DATASET_SPLIT, DATASET_ITEM_ID,
    revision=DATASET_REVISION,
)
print(f"   Artifacts:")
for art in item_detail.get("artifacts", []):
    print(f"     - path: {art.get('path')}  (media_type: {art.get('media_type', '?')})")

# Pick the first image artifact
artifacts = item_detail.get("artifacts", [])
image_artifacts = [a for a in artifacts if a.get("media_type", "").startswith("image/")]
if image_artifacts:
    DATASET_ARTIFACT_PATH = image_artifacts[0]["path"]
    print(f"\n📥 Using first image artifact: '{DATASET_ARTIFACT_PATH}'")
else:
    # Fallback: use first artifact
    DATASET_ARTIFACT_PATH = artifacts[0]["path"] if artifacts else "005.jpeg"
    print(f"\n📥 No image artifacts found, using: '{DATASET_ARTIFACT_PATH}'")

# Retrieve the image artifact
print(f"\n📥 Retrieving artifact '{DATASET_ARTIFACT_PATH}' from item '{DATASET_ITEM_ID}'...")
image_bytes = await client.get_dataset_artifact(
    DATASET_ID, DATASET_SPLIT, DATASET_ITEM_ID, DATASET_ARTIFACT_PATH,
    revision=DATASET_REVISION,
)

# Save to a temp location for downstream processing
IMAGE_PATH = PROJECT_ROOT / "temp" / Path(DATASET_ARTIFACT_PATH).name
IMAGE_PATH.parent.mkdir(parents=True, exist_ok=True)
IMAGE_PATH.write_bytes(image_bytes)

print(f"   ✅ Saved to: {IMAGE_PATH}")
print(f"   Size:       {len(image_bytes)} bytes")

🔗 Toolbox base URL: http://localhost:8002
🔗 Toolbox provider: docling

📚 Available datasets: 1
   - acessilia-dataset: Acessilia Dataset

📖 Dataset info:
   Name:        Acessilia Dataset
   Description: Reference dataset for accessibility processing pipelines. Contains PDFs, images, formulas and ground-truth annotations f...
   Splits:      ['input', 'intermediate', 'outputs']
   Items:       None

📂 Splits:
   - input: 35 items
   - intermediate: 24 items
   - outputs: 48 items

📄 Items in 'input' split (first 10):
   - 001: 6 artifacts
   - 002: 405 artifacts
   - 003: 50 artifacts
   - 004: 3 artifacts
   - 005: 3 artifacts
   - 006: 66 artifacts
   - 007: 5 artifacts
   - 008: 4 artifacts
   - 009: 4 artifacts
   - 010: 4 artifacts

🔍 Inspecting item '005' to find artifact paths...
   Artifacts:
     - path: input/005.jpeg  (media_type: image/jpeg)

📥 Using first image artifact: 'input/005.jpeg'

📥 Retrieving artifact 'input/005.jpeg' from item '005'...
   ✅ Saved to: /Users/akira

In [21]:
# ═══════════════════════════════════════════════════════════════
# Cell 4 — Disable caches for this experiment
# ═══════════════════════════════════════════════════════════════
import os

# Force bypass of remote Toolbox cache
os.environ["TOOLBOX_USE_REMOTE_CACHE"] = "false"

# Clear local filesystem cache
from backend.services.cache import clear_cache, CACHE_DIR
if CACHE_DIR.exists():
    n = await clear_cache()
    print(f"🧹 Local cache cleared: {n} file(s) removed")
else:
    print("📂 Local cache already empty")

# Verify the env var was applied
from backend.config.settings import settings
print(f"\n🔧 TOOLBOX_USE_REMOTE_CACHE now = {settings.toolbox_use_remote_cache}")
print("✅ Caches disabled for this run")

📂 Local cache already empty

🔧 TOOLBOX_USE_REMOTE_CACHE now = True
✅ Caches disabled for this run


In [25]:
# ═══════════════════════════════════════════════════════════════
# Cell 5 — Upload artifact to Toolbox
# ═══════════════════════════════════════════════════════════════
from backend.tools.toolbox_client import ToolboxClient

client = ToolboxClient()
print(f"🔗 Toolbox base URL: {client.base_url}")
print(f"🔗 Toolbox provider: {client.provider}")

artifact_id = None
if settings.toolbox_use_artifact_store:
    print("\n📤 Uploading artifact...")
    artifact_id = await client.upload_artifact(IMAGE_PATH)
    print(f"   ✅ artifact_id: {artifact_id}")
else:
    print("\n⏭️  Artifact store disabled — upload will be inline with extraction")

🔗 Toolbox base URL: http://localhost:8002
🔗 Toolbox provider: docling

📤 Uploading artifact...


2026-09-14 21:55:18.883 | INFO     | backend.tools.toolbox_client:upload_artifact:103 - Toolbox: artifact sha256:efe4501b2bf0db2a264bbd6d4be42af504be2a5cd28501bf87d0a4cfe6a7b731 armazenado (005.jpeg)


   ✅ artifact_id: sha256:efe4501b2bf0db2a264bbd6d4be42af504be2a5cd28501bf87d0a4cfe6a7b731


In [26]:
# ═══════════════════════════════════════════════════════════════
# Cell 6 — Structure extraction via Toolbox (RAW JSON, no cache)
# ═══════════════════════════════════════════════════════════════
print("🔍 Extracting structure via Toolbox (no cache)...")
print(f"   use_remote_cache=False (forced)")

result = await client.extract_structure(
    file_path=None if artifact_id else IMAGE_PATH,
    artifact_id=artifact_id,
    language="pt-BR",
    use_remote_cache=False,
)

provenance = result.get("provenance", {})
print(f"   Status:     {result.get('status')}")
print(f"   Provider:   {result.get('provider')}")
print(f"   Duration:   {provenance.get('duration_ms')} ms")
print(f"   Cache key:  {provenance.get('cache_key')}")
print(f"   Cache hit:  {provenance.get('cache_hit', 'N/A')}")

# Save raw JSON for inspection
raw_json_path = PROJECT_ROOT / "temp/raw_extraction.json"
raw_json_path.parent.mkdir(parents=True, exist_ok=True)
raw_json_path.write_text(
    json.dumps(result, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"\n📄 Raw JSON saved to: {raw_json_path}")

🔍 Extracting structure via Toolbox (no cache)...
   use_remote_cache=False (forced)


2026-09-14 21:55:54.882 | INFO     | backend.tools.toolbox_client:extract_structure:194 - Toolbox: extração concluída (19333 ms, provider=docling)


   Status:     succeeded
   Provider:   docling
   Duration:   19333 ms
   Cache key:  sha256:739d75acb24c87a61bcc86fb911cfabbabcc3f1b7dae139601cf65321a04c121
   Cache hit:  False

📄 Raw JSON saved to: /Users/akira/dados/sync/dev/acessilia/temp/raw_extraction.json


In [27]:
# ═══════════════════════════════════════════════════════════════
# Cell 7 — Inspect raw Toolbox JSON
# ═══════════════════════════════════════════════════════════════
raw = result

# General structure
print("📋 Returned JSON structure:")
print(f"   Root keys: {list(raw.keys())}")

doc = raw.get("document", {})
print(f"\n📄 document keys: {list(doc.keys())}")

elements = doc.get("elements", [])
pages = doc.get("pages", [])
print(f"\n📊 Elements: {len(elements)}")
print(f"📄 Pages:    {len(pages)}")

# Show all elements
print("\n" + "═" * 70)
print("RAW MANIFEST ELEMENTS")
print("═" * 70)
for i, el in enumerate(elements):
    el_type = el.get("type", "?")
    el_text = el.get("text", "") or ""
    el_label = el.get("raw_label", "?")
    el_page = el.get("page_number", "?")
    el_conf = el.get("confidence", "?")
    el_id = el.get("id", "?")
    text_preview = el_text[:80] if el_text else "(empty)"
    print(f"  [{i:2d}] {el_type:12s} | page={el_page} | conf={el_conf} | id={el_id}")
    print(f"        label={el_label} | text={text_preview!r}")
    if len(el_text) > 80:
        print(f"        ... ({len(el_text)} chars total)")
    print()

📋 Returned JSON structure:
   Root keys: ['status', 'capability', 'provider', 'artifacts', 'document', 'provenance']

📄 document keys: ['$schema', 'schema_version', 'manifest_id', 'revision', 'status', 'created_at', 'source', 'extractor', 'title', 'language', 'pages', 'elements', 'observations', 'obligations', 'artifacts', 'summary']

📊 Elements: 1
📄 Pages:    1

══════════════════════════════════════════════════════════════════════
RAW MANIFEST ELEMENTS
══════════════════════════════════════════════════════════════════════
  [ 0] paragraph    | page=1 | conf=None | id=element-000001
        label=text | text='Aaun'



In [28]:
# ═══════════════════════════════════════════════════════════════
# Cell 8 — Build ProcessingManifest
# ═══════════════════════════════════════════════════════════════
from datetime import datetime, timezone
from backend.core.manifest.toolbox_extractor import ToolboxExtraction
from backend.core.manifest.builder import build_processing_manifest

# Build ToolboxExtraction object
provenance = result.get("provenance", {})
extraction = ToolboxExtraction(
    document=result,
    started_at=datetime.now(timezone.utc),
    completed_at=datetime.now(timezone.utc),
    duration_ms=provenance.get("duration_ms", 0),
    version=provenance.get("provider_version", "unknown"),
    configuration={
        "ocr": True,
        "table_structure": True,
        "remote_services": True,
        "toolbox_base_url": client.base_url,
        "toolbox_provider": client.provider,
        "use_artifact_store": settings.toolbox_use_artifact_store,
        "use_remote_cache": settings.toolbox_use_remote_cache,
        "artifact_id": artifact_id,
        "cache_key": provenance.get("cache_key"),
    },
    artifact_id=artifact_id,
    cache_key=provenance.get("cache_key"),
)

print("🏗️  Building ProcessingManifest...")
manifest = build_processing_manifest(
    IMAGE_PATH,
    extraction,
    language="pt-BR",
)

print(f"\n📊 Manifest built:")
print(f"   manifest_id: {manifest.manifest_id}")
print(f"   title:       {manifest.title}")
print(f"   pages:       {len(manifest.pages)}")
print(f"   elements:    {len(manifest.elements)}")
print(f"   obligations: {len(manifest.obligations)}")
print(f"   observations:{len(manifest.observations)}")
print(f"   summary.element_types: {manifest.summary.element_types}")

🏗️  Building ProcessingManifest...

📊 Manifest built:
   manifest_id: manifest-efe4501b2bf0db2a-r1
   title:       005
   pages:       1
   elements:    1
   obligations: 0
   observations:0
   summary.element_types: {'paragraph': 1}


In [29]:
# ═══════════════════════════════════════════════════════════════
# Cell 9 — Detailed manifest inspection
# ═══════════════════════════════════════════════════════════════
print("═" * 70)
print("PROCESSING MANIFEST ELEMENTS")
print("═" * 70)

for i, el in enumerate(manifest.elements):
    text = (el.text or "").strip()
    text_preview = text[:80] if text else "(empty)"
    print(f"  [{i:2d}] {el.type:12s} | page={el.page_number} | "
          f"hier={el.hierarchy_level} | ro={el.reading_order} | "
          f"conf={el.confidence}")
    print(f"        id={el.id} | label={el.raw_label}")
    print(f"        text={text_preview!r}")
    if len(text) > 80:
        print(f"        ... ({len(text)} chars total)")
    print()

# Pages
print("═" * 70)
print("PAGES")
print("═" * 70)
for p in manifest.pages:
    print(f"  page={p.page_number} | w={p.width} | h={p.height} | "
          f"element_ids={len(p.element_ids)}")

# Obligations
print("\n" + "═" * 70)
print("OBLIGATIONS")
print("═" * 70)
for o in manifest.obligations:
    print(f"  id={o.id} | type={o.type} | status={o.status} | "
          f"targets={o.target_ids}")

══════════════════════════════════════════════════════════════════════
PROCESSING MANIFEST ELEMENTS
══════════════════════════════════════════════════════════════════════
  [ 0] paragraph    | page=1 | hier=1 | ro=1 | conf=None
        id=element-000001 | label=text
        text='Aaun'

══════════════════════════════════════════════════════════════════════
PAGES
══════════════════════════════════════════════════════════════════════
  page=1 | w=1280.0 | h=960.0 | element_ids=1

══════════════════════════════════════════════════════════════════════
OBLIGATIONS
══════════════════════════════════════════════════════════════════════


In [30]:
# ═══════════════════════════════════════════════════════════════
# Cell 10 — Test: _is_mostly_visual_manifest
# ═══════════════════════════════════════════════════════════════
import importlib
import backend.agents.pddl_orchestrator
importlib.reload(backend.agents.pddl_orchestrator)

from backend.agents.pddl_orchestrator import (
    _is_mostly_visual_manifest,
    _is_ocr_noise,
    _is_placeholder_text,
    _is_placeholder_visual_element,
    _has_meaningful_text,
)

print("🔍 _is_mostly_visual_manifest()")
is_visual = _is_mostly_visual_manifest(manifest)
print(f"   Result: {is_visual}")
print(f"   (True = manifest is mostly visual)")
print()

print("═" * 70)
print("BREAKDOWN: meaningful text vs placeholder vs OCR noise")
print("═" * 70)

for i, el in enumerate(manifest.elements):
    text = (el.text or "").strip()
    if not text:
        continue
    
    is_noise = _is_ocr_noise(text)
    is_ph = _is_placeholder_text(text)
    meaningful = not is_noise and not is_ph
    
    # Only elements that affect _is_mostly_visual_manifest
    if el.type in {"heading", "title", "paragraph", "list_item", "table", "code", "formula"}:
        flag = "🟢 MEANINGFUL" if meaningful else (
            "🟡 PLACEHOLDER" if is_ph else "🔴 OCR NOISE"
        )
        print(f"  [{i:2d}] {el.type:12s} | {flag} | text={text[:60]!r}")
    else:
        flag = "🔵 IGNORED" if not meaningful else "🟢 MEANINGFUL"
        print(f"  [{i:2d}] {el.type:12s} | {flag} | text={text[:60]!r}")

🔍 _is_mostly_visual_manifest()
   Result: True
   (True = manifest is mostly visual)

══════════════════════════════════════════════════════════════════════
BREAKDOWN: meaningful text vs placeholder vs OCR noise
══════════════════════════════════════════════════════════════════════
  [ 0] paragraph    | 🔴 OCR NOISE | text='Aaun'


In [31]:
# ═══════════════════════════════════════════════════════════════
# Cell 11 — Run _enrich_picture_descriptions (VisionAgent)
# ═══════════════════════════════════════════════════════════════
import importlib
import backend.agents.pddl_orchestrator
importlib.reload(backend.agents.pddl_orchestrator)

from backend.agents.pddl_orchestrator import (
    _enrich_picture_descriptions,
    _is_mostly_visual_manifest,
    _is_placeholder_visual_element,
    _has_meaningful_text,
)

print("🔍 Before enrichment:")
pictures = [el for el in manifest.elements if el.type == "picture"]
placeholders = [el for el in manifest.elements if _is_placeholder_visual_element(el)]
print(f"   pictures: {len(pictures)}")
print(f"   placeholders: {len(placeholders)}")
print(f"   _is_mostly_visual_manifest: {_is_mostly_visual_manifest(manifest)}")
print(f"   current text: {manifest.elements[0].text!r}")
print()

print("🚀 Running _enrich_picture_descriptions...")
await _enrich_picture_descriptions(
    manifest,
    IMAGE_PATH,
    mode="medio",
)

print()
print("🔍 After enrichment:")
for el in manifest.elements:
    text = (el.text or "").strip()
    print(f"   id={el.id} | type={el.type} | text={text[:100]!r}")
    if len(text) > 100:
        print(f"        ... ({len(text)} chars total)")

2026-09-14 22:59:41.857 | DEBUG    | backend.agents.vision_agent:describe_region:69 - [page 1] Sending region to vision (215398 bytes, type=embedded_image)


🔍 Before enrichment:
   pictures: 0
   placeholders: 1
   _is_mostly_visual_manifest: True
   current text: 'Aaun'

🚀 Running _enrich_picture_descriptions...


2026-09-14 22:59:53.039 | INFO     | backend.agents.pddl_orchestrator:_enrich_picture_descriptions:364 - Pipeline PDDL: 1 image(s) enriched with visual description



🔍 After enrichment:
   id=element-000001 | type=picture | text='Céu com tonalidades de laranja e rosa, com uma leve neblina ou poluição atmosférica. No horizonte, s'
        ... (571 chars total)


In [32]:
# ═══════════════════════════════════════════════════════════════
# Cell 12 — Render final text
# ═══════════════════════════════════════════════════════════════
from backend.agents.pddl_orchestrator import _manifest_pages_to_payload, _render_text_from_pages

print("📝 Rendering final text...")
pages_payload = _manifest_pages_to_payload(manifest)
text_output = _render_text_from_pages(pages_payload)

print(f"\n📊 Pages in payload: {len(pages_payload)}")
for p in pages_payload:
    print(f"   page={p['page_number']} | {len(p['text'])} chars | {len(p['blocks'])} blocks")

print("\n" + "═" * 70)
print("FINAL RENDERED TEXT")
print("═" * 70)
print(text_output[:500] if text_output else "(empty)")
if len(text_output) > 500:
    print(f"\n... ({len(text_output)} chars total)")

print("\n" + "═" * 70)
print("REPR (first 200 chars)")
print("═" * 70)
print(repr(text_output[:200]))

📝 Rendering final text...

📊 Pages in payload: 1
   page=1 | 571 chars | 1 blocks

══════════════════════════════════════════════════════════════════════
FINAL RENDERED TEXT
══════════════════════════════════════════════════════════════════════
=== Pagina 1 ===
Céu com tonalidades de laranja e rosa, com uma leve neblina ou poluição atmosférica. No horizonte, silhuetas de edifícios urbanos de diferentes alturas e formas, incluindo prédios residenciais e comerciais. Alguns prédios apresentam antenas ou estruturas no topo. Na parte inferior da imagem, silhuetas escuras de árvores e vegetação, que ocupam os cantos esquerdo e direito da composição, parcialmente obscurecendo a visão dos edifícios mais próximos. A iluminação é difusa, caracte

... (588 chars total)

══════════════════════════════════════════════════════════════════════
REPR (first 200 chars)
══════════════════════════════════════════════════════════════════════
'=== Pagina 1 ===\nCéu com tonalidades de laranja e rosa, com uma

In [33]:
# ═══════════════════════════════════════════════════════════════
# Cell 14 — Full diagnostic summary
# ═══════════════════════════════════════════════════════════════
print("=" * 70)
print("FULL DIAGNOSTIC SUMMARY")
print("=" * 70)
print()

# 1. Check if final text is garbage
text_output = _render_text_from_pages(_manifest_pages_to_payload(manifest))
if len(text_output.strip()) < 10:
    print("❌ PROBLEM: Final text too short or empty")
    print(f"   Size: {len(text_output.strip())} chars")
    print(f"   Content: {text_output[:100]!r}")
else:
    print(f"✅ Final text OK: {len(text_output.strip())} chars")
print()

# 2. Check Toolbox extraction
elements_raw = result.get("document", {}).get("elements", [])
if not elements_raw:
    print("❌ PROBLEM: Toolbox returned 0 elements")
else:
    texts = [e.get("text", "") for e in elements_raw if e.get("text")]
    if all(len(t.strip()) < 5 for t in texts):
        print("❌ PROBLEM: Toolbox returned only very short texts (OCR noise)")
        for e in elements_raw:
            print(f"   type={e.get('type')} text={e.get('text', '')!r}")
    else:
        print("✅ Toolbox returned texts with reasonable length")
print()

# 3. Check remote cache (disabled in this run)
cache_hit = result.get("provenance", {}).get("cache_hit")
cache_key = result.get("provenance", {}).get("cache_key")
if cache_hit:
    print(f"⚠️  Toolbox used REMOTE CACHE (cache_key={cache_key})")
    print("   (even with no_cache=True — may be a Toolbox-side log)")
else:
    print("✅ Toolbox did NOT use remote cache (fresh extraction as configured)")
print()

# 4. Check image enrichment
is_visual = _is_mostly_visual_manifest(manifest)
pictures = [el for el in manifest.elements if el.type == "picture"]
print(f"📸 _is_mostly_visual_manifest: {is_visual}")
print(f"📸 'picture' elements: {len(pictures)}")
if not is_visual and not pictures:
    print("⚠️  Manifest is not visual and has no pictures — VisionAgent won't be called")
    print("   Garbage OCR text goes straight to output!")
elif is_visual:
    print("✅ Manifest is visual — VisionAgent will be called to describe")
print()

# 5. Summary
print("═" * 70)
print("RECOMMENDATIONS")
print("═" * 70)
if len(text_output.strip()) < 10:
    print("1. Local and remote caches are already disabled in this run.")
    print("2. If Toolbox still returned garbage, the issue is in Docling/OCR.")
    print("3. Check which provider the Toolbox is using (docling or other).")
    print("4. Consider improving _is_ocr_noise() to catch more OCR garbage patterns.")

FULL DIAGNOSTIC SUMMARY

✅ Final text OK: 588 chars

❌ PROBLEM: Toolbox returned only very short texts (OCR noise)
   type=paragraph text='Aaun'

✅ Toolbox did NOT use remote cache (fresh extraction as configured)

📸 _is_mostly_visual_manifest: True
📸 'picture' elements: 1
✅ Manifest is visual — VisionAgent will be called to describe

══════════════════════════════════════════════════════════════════════
RECOMMENDATIONS
══════════════════════════════════════════════════════════════════════


In [34]:
# ═══════════════════════════════════════════════════════════════
# Cell 15 — Utility: clear local cache (already done in cell 4)
# ═══════════════════════════════════════════════════════════════
from backend.services.cache import clear_cache

print("🧹 Local cache was already cleared in cell 4 of this run.")
print()
print("If you need to clear again in another run:")
print("   await clear_cache()")
print()
print("Or via terminal:")

🧹 Local cache was already cleared in cell 4 of this run.

If you need to clear again in another run:
   await clear_cache()

Or via terminal:
